# 状態予測モデル

状態予測モデルでは、未来予測を使った意思決定を、実装可能な形に分解して学びます。


## 背景と目的

将来状態を予測できると、行動前に結果を見積もって失敗を減らせます。

予測モデルを理解すると、計画と異常検知の両方に応用できます。

1ステップ/多ステップ誤差の違いを確認します。


## 最初に解きたい疑問

1. `状態予測モデル` は、観測予測モデルとどこが違うのか。
2. 1ステップ予測が当たっても、なぜ長期では崩れるのか。
3. `rollout` は何ステップ先まで見ているのか。
4. `mean_error` だけ見て、何を見落としやすいのか。
5. 異常検知に使うなら、どの誤差を閾値にするのか。


## 先に押さえる言葉

- `未来状態予測`: 次の状態をあらかじめ推定することです。行動の結果を前もって見るために使います。
- `誤差累積`: 小さい誤差が何ステップも重なって大きくなることです。長期予測が崩れる典型的な原因です。
- `ロールアウト`: 予測を連続して進めることです。1回の予測では見えない崩れ方を確かめられます。
- `異常検知`: 予測と実測のずれから、いつもと違う状態を見つけることです。予測モデルの応用先の一つです。


## 実行前の見取り図

1. `z_next` がまず 1 ステップ分の予測として出るか。
2. `rollout = [...]` の列が途中で不自然に発散していないか。
3. `mean_error` と個々の `errors` が両方確認できるか。


## 出力の読み方

- `mean_error` と個別 `errors` のどちらを優先して見るか。
- `rollout` の崩れ方で何を疑うか。


## つまずきやすい点

- 1ステップ誤差と多ステップ誤差を分けて見る意味。
- 異常検知にどうつなげるかの実務的な説明。
- 1ステップ誤差と多ステップ誤差を分ける意味。
- 異常検知につなげる流れ。


## このノートの守備範囲

このノートでは次の点は入口だけ触れるか、別ノートに分けて扱います。

- ここでの予測は toy rollout であり、実務の長期予測評価そのものではない点。


## 実験 1: 状態予測モデルの初期遷移

未来状態予測の誤差を見るため、遷移初期値を定義します。


In [ ]:
z_t = -0.05
a_t = 1.1
A, B = 0.89, 0.21
z_next = A * z_t + B * a_t
print('task = state-prediction-models')
print('z_next =', round(z_next, 6))

この遷移誤差を後続で時系列的に評価します。



## 実験 2: 観測予測を作る

次に、潜在状態から観測を復元する写像を作ります。状態推定と観測再現の役割分担をコードで掴みます。


In [ ]:
def decode(z):
    return {'position': 2.5 * z + 0.1, 'velocity': 0.8 * z - 0.05}
obs_next = decode(z_next)
print('obs_next =', {k: round(v, 4) for k, v in obs_next.items()})
print('keys =', list(obs_next.keys()))

観測予測を別関数に切ると、遷移誤差と観測誤差を分離して調整できます。



## 式と実装の往復

1. $z_{t+1} = f_{\theta}(z_t, a_t)$
2. $\hat{o}_{t+1} = g_{\theta}(z_{t+1})$


## 実験 3: ロールアウトを試す

ここで複数ステップ予測を実行します。1ステップでは見えない誤差累積を把握するためです。


In [ ]:
actions = [0.0, 1.0, 1.0, 0.0, -0.5]
z = 0.1
traj = []
for a in actions:
    z = 0.92 * z + 0.18 * a
    traj.append(round(z, 5))
print('rollout =', traj)

長期予測で崩れるなら、遷移モデルの安定性や状態表現の情報量不足を疑います。



## 実験 4: 計画候補を比較する

次に、複数の行動列を比較して、どの計画が望ましいかを評価します。モデルベース強化学習の中心操作です。


In [ ]:
plans = [[0, 1, 1], [1, 1, 1], [0, 0, 1]]
def score_plan(plan):
    z = 0.1
    for a in plan:
        z = 0.92 * z + 0.18 * a
    return z
scores = [round(score_plan(p), 5) for p in plans]
print('scores =', scores)

計画評価が可能になると、実環境での試行回数を抑えた探索がしやすくなります。



## 実験 5: モデル誤差を監視する

最後に、予測と実測の差を定量化します。世界モデルは『予測できる範囲』を常に点検する運用が重要です。


In [ ]:
pred = [0.10, 0.22, 0.31, 0.29]
real = [0.11, 0.25, 0.28, 0.35]
errors = [abs(p - r) for p, r in zip(pred, real)]
print('errors =', [round(e, 4) for e in errors])
print('mean_error =', round(sum(errors) / len(errors), 5))

平均誤差だけでなく時点別誤差を追うと、どの遷移条件でモデルが弱いかを特定しやすくなります。異常検知に使うなら、たとえば `mean_error` ではなく各時点誤差の 95 パーセンタイルや固定閾値を超えた時点を異常候補とみなします。


## 振り返り

今回のノートで押さえておくべき誤解しやすい点を整理します。

1. 1ステップ誤差だけで安心してしまう
2. 長期予測の誤差爆発を監視しない
3. 状態表現が不足しているのにモデルだけ調整する
